# 007 · Comprehensions

> **Syntax › Collections** · Python 3.14

A *comprehension* builds a new list, dict or set from an iterable in a single expression: `[x * 2 for x in nums if x > 0]`. For a C# developer, it is Python's built-in syntax for the most common LINQ pipelines — `Select`, `Where`, `SelectMany`, `ToList`, `ToDictionary`, `ToHashSet`.

**In this notebook you will learn:**

- list, dict and set comprehensions and their LINQ equivalents
- filtering with `if` and choosing values with a conditional expression
- nested comprehensions (the `SelectMany` of Python) and when they get unreadable
- generator expressions — the lazy version
- how variables are scoped inside a comprehension
- when a plain `for` loop is the better choice

## C# mental model

```csharp
// C#
var squares = numbers.Where(n => n % 2 == 0).Select(n => n * n).ToList();
```

```python
# Python
squares = [n * n for n in numbers if n % 2 == 0]
```

Read a comprehension as: **[ *output expression* `for` *variable* `in` *source* `if` *filter* ]**. Note the order: the projection (`Select`) comes **first**, the source and filter (`Where`) after — the opposite of LINQ method syntax, but close to LINQ *query* syntax read backwards (`from n in numbers where ... select n * n`).

## List comprehensions — `Select(...).ToList()`

In [ ]:
numbers = [1, 2, 3, 4, 5]

squares = [n * n for n in numbers]              # Select(n => n * n).ToList()
words = ["apple", "Banana", "cherry"]
upper = [w.upper() for w in words]              # Select(w => w.ToUpper()).ToList()
lengths = [len(w) for w in words]

print(squares, upper, lengths)

Compare with the equivalent loop. The comprehension does the same thing, is shorter, and is usually faster:

In [ ]:
squares_loop = []
for n in numbers:
    squares_loop.append(n * n)

print(squares_loop == [n * n for n in numbers])   # True

## Filtering with `if` — `Where`

A trailing `if` keeps only items for which the condition is true.

```csharp
// C#
var evens = numbers.Where(n => n % 2 == 0).ToList();
var longWords = words.Where(w => w.Length > 5).Select(w => w.ToLower()).ToList();
```

In [ ]:
numbers = list(range(1, 11))
words = ["apple", "Banana", "cherry", "fig"]

evens = [n for n in numbers if n % 2 == 0]
long_words = [w.lower() for w in words if len(w) > 5]
print(evens)
print(long_words)

Multiple `if` clauses are combined with **and**:

In [ ]:
print([n for n in range(50) if n % 3 == 0 if n % 5 == 0])   # same as `if n % 3 == 0 and n % 5 == 0`

## Conditional expressions in the output

To *transform* every item differently (rather than drop some), put a conditional expression `a if cond else b` in the **output** position. This is C#'s ternary `cond ? a : b` inside a `Select`.

```csharp
// C#
var labels = numbers.Select(n => n % 2 == 0 ? "even" : "odd").ToList();
```

In [ ]:
numbers = [1, 2, 3, 4]
labels = ["even" if n % 2 == 0 else "odd" for n in numbers]
print(labels)

⚠️ **Gotcha:** the position of `if` matters.

- `[x for x in xs if cond]` — **filter** (`Where`); there is no `else`.
- `[a if cond else b for x in xs]` — **map** with a choice (`Select` with `?:`); `else` is required.

Writing `[x for x in xs if cond else y]` is a `SyntaxError`.

In [ ]:
temps = [12, -3, 25, -8]

only_positive = [t for t in temps if t > 0]           # filter: 2 items
clamped = [t if t > 0 else 0 for t in temps]          # map: still 4 items
print(only_positive, clamped)

## Dict comprehensions — `ToDictionary`

Use `{key: value for ...}`.

```csharp
// C#
var lengths = words.ToDictionary(w => w, w => w.Length);
var byId = products.ToDictionary(p => p.Id);
```

In [ ]:
words = ["apple", "fig", "cherry"]
lengths = {w: len(w) for w in words}
print(lengths)

prices = {"tea": 3.0, "coffee": 4.5, "juice": 5.0}
discounted = {item: round(price * 0.9, 2) for item, price in prices.items()}
expensive = {item: price for item, price in prices.items() if price > 4}
print(discounted)
print(expensive)

⚠️ **Gotcha:** C#'s `ToDictionary` **throws** on duplicate keys. A dict comprehension silently keeps the **last** value for each key:

In [ ]:
names = ["alice", "adam", "bob"]
by_initial = {n[0]: n for n in names}
print(by_initial)   # {'a': 'adam', 'b': 'bob'} — 'alice' was overwritten

Swapping keys and values is a classic one-liner:

In [ ]:
codes = {"red": 1, "green": 2, "blue": 3}
names_by_code = {v: k for k, v in codes.items()}
print(names_by_code)

## Set comprehensions — `ToHashSet`

Use `{expr for ...}` (no colon). Duplicates are removed automatically.

```csharp
// C#
var domains = emails.Select(e => e.Split('@')[1]).ToHashSet();
```

In [ ]:
emails = ["alice@example.com", "bob@example.org", "carol@example.com"]
domains = {e.split("@")[1] for e in emails}
print(sorted(domains))

## Nested comprehensions — `SelectMany`

Multiple `for` clauses are nested loops, written **in the same order** as the equivalent `for` statements. The classic use is flattening — C#'s `SelectMany`.

```csharp
// C#
var flat = matrix.SelectMany(row => row).ToList();
var pairs = (from x in xs from y in ys select (x, y)).ToList();
```

In [ ]:
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

flat = [n for row in matrix for n in row]     # outer loop first, then inner
print(flat)

# exactly equivalent to:
flat_loop = []
for row in matrix:
    for n in row:
        flat_loop.append(n)
print(flat == flat_loop)

In [ ]:
sizes = ["S", "M"]
colours = ["red", "blue"]
variants = [(s, c) for s in sizes for c in colours]   # cartesian product
print(variants)

A comprehension inside the *output* position builds a nested structure — e.g. a 2-D grid or a transpose:

In [ ]:
grid = [[0] * 3 for _ in range(2)]          # 2 rows x 3 columns, independent rows
print(grid)

matrix = [[1, 2, 3], [4, 5, 6]]
transposed = [[row[i] for row in matrix] for i in range(3)]
print(transposed)

⚠️ **Readability guidance:** one `for` with an optional `if` is ideal. Two `for` clauses are fine for simple flattening. Beyond that — three loops, multiple conditions, or nested comprehensions inside comprehensions — switch to regular loops or helper functions. Code is read far more often than it is written.

In [ ]:
# Hard to read — avoid:
result = [x * y for x in range(5) if x % 2 for y in range(5) if y > x if (x + y) % 3]

# Same logic, clear:
result_loop = []
for x in range(5):
    if x % 2 == 0:
        continue
    for y in range(x + 1, 5):
        if (x + y) % 3:
            result_loop.append(x * y)

print(result == result_loop, result)

## Generator expressions — the lazy version

Replace the square brackets with **parentheses** and you get a *generator expression*: it produces items **lazily**, one at a time, instead of building a list. This is closer to a LINQ query *before* you call `ToList()` — deferred execution.

When a generator expression is the only argument of a function, the extra parentheses can be dropped:

```csharp
// C#
var total = orders.Where(o => o.Paid).Sum(o => o.Amount);   // no intermediate list
```

In [ ]:
numbers = range(1, 1_000_001)

total = sum(n * n for n in numbers)         # no million-item list is built
print(total)

print(any(n > 999_999 for n in numbers))    # stops at the first match — like LINQ Any()
lazy = (n * 2 for n in [1, 2, 3])
print(lazy)                                 # a generator object, not a list
print(list(lazy))

⚠️ **Gotcha:** a generator can only be consumed **once** — unlike an `IEnumerable<T>` query, which re-runs every time you enumerate it.

In [ ]:
gen = (n * n for n in range(4))
print(list(gen))    # [0, 1, 4, 9]
print(list(gen))    # [] — already exhausted

Generators and lazy evaluation get a full treatment in *Iterators & Generators › 003 · Generator Expressions and Lazy Evaluation*.

## Variable scoping

The loop variable of a comprehension is **local to the comprehension** — it does not leak into the surrounding scope, and it doesn't overwrite an outer variable with the same name. (A plain `for` loop *does* leave its variable behind.)

In [ ]:
x = "outer"
squares = [x * x for x in range(3)]
print(x)            # 'outer' — untouched

for y in range(3):
    pass
print(y)            # 2 — a for loop's variable survives the loop

The one exception is the *walrus* operator `:=`, which deliberately binds in the **enclosing** scope. That makes it useful for capturing a value computed during filtering:

In [ ]:
data = [3, 8, 1, 9, 4]

big = [doubled for n in data if (doubled := n * 2) > 10]
print(big)
print(doubled)      # the walrus target is visible after the comprehension (last value: 8)

⚠️ **Gotcha:** comprehensions are for building collections, not for side effects. `[print(x) for x in items]` works but creates a throwaway list of `None`s. Use a `for` loop when you only want the effect.

In [ ]:
# Don't:
_ = [print(x) for x in ["a", "b"]]

# Do:
for x in ["a", "b"]:
    print(x)

## When to use a plain loop instead

Prefer a regular `for` loop when:

- the body has **side effects** (printing, writing files, mutating other objects)
- you need **multiple statements**, `try`/`except`, or `break`
- the comprehension needs more than ~2 clauses or doesn't fit comfortably on one or two lines
- you are building **several** results at once

In [ ]:
orders = [("alice", 120), ("bob", -5), ("carol", 80), ("dave", 300)]

valid, rejected = [], []
for customer, amount in orders:          # two outputs + logic: a loop is clearer
    if amount <= 0:
        rejected.append(customer)
        continue
    valid.append((customer, amount))

print(valid)
print(rejected)

## LINQ → Python mapping

| LINQ | Python |
|---|---|
| `xs.Select(x => f(x)).ToList()` | `[f(x) for x in xs]` |
| `xs.Where(x => p(x)).ToList()` | `[x for x in xs if p(x)]` |
| `xs.Where(p).Select(f).ToList()` | `[f(x) for x in xs if p(x)]` |
| `xs.Select(x => c ? a : b)` | `[a if c else b for x in xs]` |
| `xs.SelectMany(x => x.Items)` | `[i for x in xs for i in x.items]` |
| `xs.ToDictionary(x => k(x), x => v(x))` | `{k(x): v(x) for x in xs}` |
| `xs.ToHashSet()` / `Select(f).ToHashSet()` | `set(xs)` / `{f(x) for x in xs}` |
| `xs.Select(f)` (deferred) | `(f(x) for x in xs)` |
| `xs.Sum(x => f(x))` | `sum(f(x) for x in xs)` |
| `xs.Any(x => p(x))` / `All` | `any(p(x) for x in xs)` / `all(...)` |
| `xs.Count(x => p(x))` | `sum(1 for x in xs if p(x))` |
| `xs.Select((x, i) => ...)` | `[... for i, x in enumerate(xs)]` |
| `xs.Zip(ys, (x, y) => ...)` | `[... for x, y in zip(xs, ys)]` |
| `xs.Distinct()` | `list(dict.fromkeys(xs))` |
| `xs.OrderBy(k)` | `sorted(xs, key=k)` |
| `xs.Max(k)` / `MaxBy(k)` | `max(map(k, xs))` / `max(xs, key=k)` |

In [ ]:
# a few rows from the table, run for real
xs = [3, 1, 4, 1, 5, 9, 2, 6]
ys = "abcdefgh"

print(sum(1 for x in xs if x > 3))                 # Count(x => x > 3)
print(all(x > 0 for x in xs))                      # All(x => x > 0)
print([f"{i}:{x}" for i, x in enumerate(xs[:3])])  # Select((x, i) => ...)
print([f"{y}{x}" for x, y in zip(xs, ys)][:3])     # Zip
print(max(["fig", "banana", "kiwi"], key=len))     # MaxBy(w => w.Length)

## Summary

| Task | C# (LINQ) | Python |
|---|---|---|
| Project | `Select(f).ToList()` | `[f(x) for x in xs]` |
| Filter | `Where(p).ToList()` | `[x for x in xs if p(x)]` |
| Choose per item | `Select(x => c ? a : b)` | `[a if c else b for x in xs]` |
| Flatten | `SelectMany` | `[y for x in xs for y in x]` |
| Build dict | `ToDictionary(k, v)` | `{k: v for x in xs}` |
| Build set | `ToHashSet()` | `{f(x) for x in xs}` |
| Deferred/lazy | un-materialised query | `(f(x) for x in xs)` |
| Duplicate dict keys | `ToDictionary` throws | last value wins |
| Variable scope | lambda parameter scope | local to the comprehension (except `:=`) |

## Exercises

1. From `words = ["Python", "is", "a", "great", "language", "to", "learn"]`, build a list of the **uppercased** words with more than 2 characters.
2. Build a dict mapping each number 1–5 to its cube, then a set of the last digits of those cubes.
3. Flatten `nested = [[1, 2], [3], [], [4, 5, 6]]` and keep only the odd numbers, in one comprehension.
4. Given `orders = [("alice", 30), ("bob", 0), ("alice", 15), ("carol", 50)]`, compute the total amount of all non-zero orders using a generator expression, and the set of customers who placed them.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
words = ["Python", "is", "a", "great", "language", "to", "learn"]
result = [w.upper() for w in words if len(w) > 2]
# ['PYTHON', 'GREAT', 'LANGUAGE', 'LEARN']

# 2
cubes = {n: n ** 3 for n in range(1, 6)}       # {1: 1, 2: 8, 3: 27, 4: 64, 5: 125}
last_digits = {c % 10 for c in cubes.values()} # {1, 8, 7, 4, 5}

# 3
nested = [[1, 2], [3], [], [4, 5, 6]]
odds = [n for sub in nested for n in sub if n % 2]   # [1, 3, 5]

# 4
orders = [("alice", 30), ("bob", 0), ("alice", 15), ("carol", 50)]
total = sum(amount for _, amount in orders if amount > 0)         # 95
customers = {name for name, amount in orders if amount > 0}       # {'alice', 'carol'}
```

</details>